# LLM 12일차 (3) — 서울시 상권 분석 2단계 : PydanticAI + LangGraph 4단계 분석 에이전트 (재구성 → NL2SQL → 실행 → 보고서)

교재 `AI Agent` **p.148~149** (개발 시나리오 2단계 — 핵심 로직) 와 수업 필기 `data_analysis_agent` 를 정리 · 완성한 노트입니다. 필기는 **출력 자료형 3개 · 상태 · 도구 함수 2개까지 만들고 노드 함수는 주석만** 있어서, 교재의 4단계 흐름대로 노드 · 그래프 · 체크포인터 실행까지 채웠습니다.

```
START → rephrase_node ─ is_data_query=True ─→ sql_node (NL2SQL) → execute_node (MySQL) → report_node → END
        (RephraseResult)  └ False ───────────────────────────────────────────→ general_node ──┘
                                             (AIOMySQLSaver 체크포인터 · thread_id)
```

> **출력 읽는 법** — `# ⏳` 진짜 gpt-5-nano 셀(비어 있음) · `# 🧪` LLM 자리에 **규칙으로 답하는 `FunctionModel`** 을 넣어 실제로 돌린 셀 · 표시 없음 = 실제 실행. 테이블은 상권 노트에서 만든 **🧪 모의 데이터 72행**(숫자는 실제 서울시 데이터가 아님) · MySQL 은 MariaDB 10.11. **가짜 모델의 SQL 은 정해 둔 질문에만 맞는다** — 진짜 NL2SQL 품질은 ⏳ 셀로 확인해야 한다.

## 0. 설치 · 실행 방법

`pip install "pydantic-ai-slim[openai]" langgraph langgraph-checkpoint-mysql aiomysql python-dotenv` · MySQL 에 `seoul_commercial` 테이블([`seoul_commercial_db_pipeline.ipynb`](./seoul_commercial_db_pipeline.ipynb)) · `.env` 에 `OPENAI_API_KEY` · `MYSQL_*` (📝 SQL 실행용으로 **SELECT 만 가능한 계정** `MYSQL_RO_USER` · `MYSQL_RO_PASSWORD` 를 따로 두기를 권장 — §3)

In [1]:
import os, re, asyncio
os.environ.setdefault('PYDANTIC_AI_NO_BANNER', '1')
from decimal import Decimal
from typing import Annotated, Literal
import aiomysql
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.function import FunctionModel, AgentInfo
from pydantic_ai.messages import ModelResponse, ToolCallPart, TextPart
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.mysql.aio import AIOMySQLSaver

load_dotenv(find_dotenv(usecwd=True), override=True)
MODEL_NAME = 'openai:gpt-5-nano'          # 📝 2026-12-11 지원 종료 예정 → 이후에는 다른 모델 이름으로

In [2]:
# 접속 정보는 .env 에서 (비밀번호를 코드 · 출력에 남기지 않는다)
MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = int(os.getenv('MYSQL_PORT', '3306'))
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', '')
MYSQL_DB = os.getenv('MYSQL_DB', 'langgraph_db')
TABLE_NAME = 'seoul_commercial'
print(f'{MYSQL_USER}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB} (비밀번호 {"있음" if MYSQL_PASSWORD else "없음"})')

lg@localhost:3306/langgraph_db (비밀번호 있음)


## 1. 출력 자료형 · 상태 (필기)

In [3]:
class RephraseResult(BaseModel):
    rephrased_query: str = Field(description='맥락이 반영된 구체적인 질문')
    is_data_query: bool = Field(description='데이터 조회가 필요한 질문인지 여부')

class SQLResult(BaseModel):
    sql: str = Field(description='실행 가능한 SQL 쿼리')
    explanation: str | None = Field(default=None, description='쿼리에 대한 짧은 설명')

class ReportResult(BaseModel):
    title: str = Field(description='보고서 제목')
    content: str = Field(description='마크다운 형식의 분석 본문')
    conclusion: str = Field(description='핵심 요약 및 제언')

class AnalysisState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    rephrased_query: str = ''
    is_data_query: bool = True
    sql_query: str = ''
    sql_result: list[dict] = []
    final_report: ReportResult | None = None

## 2. 도구 함수 — 스키마 조회 · SQL 실행 (필기 + 📝 고친 부분)

In [4]:
async def connect(user=None, password=None, **kw):
    return await aiomysql.connect(host=MYSQL_HOST, port=MYSQL_PORT, user=user or MYSQL_USER,
                                  password=MYSQL_PASSWORD if password is None else password,
                                  db=MYSQL_DB, charset='utf8mb4', **kw)

async def get_db_schema_info() -> str:
    conn = None
    try:
        conn = await connect()
        async with conn.cursor() as cursor:
            await cursor.execute(f'SHOW CREATE TABLE `{TABLE_NAME}`;')
            result = await cursor.fetchone()
            return result[1] if result else '테이블 정보를 찾을 수 없습니다.'
    except Exception as e:
        return f'스키마 조회 오류: {e}'
    finally:
        if conn: conn.close()                    # 📝 필기는 연결을 닫지 않았다 (호출할 때마다 연결이 쌓인다)

schema = await get_db_schema_info()
print(schema[:330], '…', f'({schema.count(chr(10))}줄)')

CREATE TABLE `seoul_commercial` (
  `STDR_YYQU_CD` bigint(20) DEFAULT NULL,
  `TRDAR_SE_CD` text DEFAULT NULL,
  `TRDAR_SE_CD_NM` text DEFAULT NULL,
  `TRDAR_CD` bigint(20) DEFAULT NULL,
  `TRDAR_CD_NM` text DEFAULT NULL,
  `SVC_INDUTY_CD` text DEFAULT NULL,
  `SVC_INDUTY_CD_NM` text DEFAULT NULL,
  `THSMON_SELNG_AMT` bigint(20) … (56줄)


In [5]:
def to_number(v):
    # MySQL 집계(SUM 등)는 Decimal 로 온다 → 정수면 int, 아니면 float (필기)
    if isinstance(v, Decimal):
        return int(v) if v == int(v) else float(v)
    return v

FORBIDDEN = re.compile(r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|TRUNCATE|REPLACE|GRANT|REVOKE|RENAME)\b', re.I)

def check_sql(sql: str) -> str | None:
    # 📝 LLM 이 만든 SQL 은 실행 전에 검사 : SELECT 한 문장만
    s = sql.strip().rstrip(';').strip()
    if not s.upper().startswith(('SELECT', 'WITH')):  return 'SELECT 문만 실행할 수 있습니다.'
    if ';' in s:                                       return '한 번에 한 문장만 실행할 수 있습니다.'
    if FORBIDDEN.search(s):                            return '데이터를 바꾸는 명령은 실행할 수 없습니다.'
    return None

RO_USER, RO_PASSWORD = os.getenv('MYSQL_RO_USER'), os.getenv('MYSQL_RO_PASSWORD')

async def execute_sql_query(sql: str) -> list[dict] | str:
    if (problem := check_sql(sql)):
        return f'SQL 실행 거부: {problem}'
    conn = None
    try:
        conn = await connect(user=RO_USER, password=RO_PASSWORD, cursorclass=aiomysql.DictCursor)   # 📝 읽기 전용 계정
        async with conn.cursor() as cursor:
            await cursor.execute(sql)
            rows = await cursor.fetchall()
            return [{k: to_number(v) for k, v in row.items()} for row in rows]
    except Exception as e:
        return f'SQL 실행 오류: {e}'
    finally:
        if conn: conn.close()

r = await execute_sql_query(f'SELECT STDR_YYQU_CD, SUM(THSMON_SELNG_AMT) AS total FROM {TABLE_NAME} GROUP BY STDR_YYQU_CD ORDER BY 1')
print('분기별 합계 :', r[:3], '…', '| total 자료형 :', type(r[0]['total']).__name__)
print('DROP 시도   :', await execute_sql_query(f'DROP TABLE {TABLE_NAME}'))

분기별 합계 : [{'STDR_YYQU_CD': 20251, 'total': 22717034370}, {'STDR_YYQU_CD': 20252, 'total': 24209240374}, {'STDR_YYQU_CD': 20253, 'total': 19547440583}] … | total 자료형 : int
DROP 시도   : SQL 실행 거부: SELECT 문만 실행할 수 있습니다.


In [6]:
# 검사를 통과하는 '교묘한' SQL 이라도 읽기 전용 계정이면 DB 가 막는다
if RO_USER:                                             # 읽기 전용 계정을 설정한 경우에만 시험 (아니면 정말 지워질 수 있다)
    conn = await connect(user=RO_USER, password=RO_PASSWORD)
    try:
        async with conn.cursor() as cur:
            await cur.execute(f'DELETE FROM {TABLE_NAME} LIMIT 1')
    except Exception as e:
        print('읽기 전용 계정으로 DELETE →', type(e).__name__, str(e)[:90])
    finally:
        conn.close()
else:
    print('MYSQL_RO_USER 가 없어 건너뜀')

읽기 전용 계정으로 DELETE → OperationalError (1142, "DELETE command denied to user 'reader'@'localhost' for table `langgraph_db`.`seoul


- 📝 필기는 LLM 이 쓴 SQL 을 **root 계정으로 그대로 실행**한다. NL2SQL 에서 LLM 이 `DROP` · `DELETE` 를 만들면(또는 사용자가 그렇게 유도하면) 데이터가 사라진다. ① 실행 전 검사(`check_sql`) ② **SELECT 권한만 있는 계정**(`GRANT SELECT ON langgraph_db.* TO reader`) 두 겹으로 막았다
- `SUM` 결과가 `Decimal` 로 오는 것은 실제로 확인됐다 → `to_number` 로 int · float 변환 (필기 그대로)

## 3. PydanticAI Agent 4개 · 노드 (필기가 주석만 남긴 곳)

📝 NL2SQL 에는 `SHOW CREATE TABLE` 만으로는 부족하다 — 영문 열 이름(`FML_SELNG_AMT`)만 보고는 '여성 매출' 인지 알 수 없다. 그래서 SQL 프롬프트에 **한글 뜻(`column_mapping`)** 도 함께 넣었다.

In [7]:
import ast
column_mapping = ast.literal_eval(open('download.txt', encoding='utf-8').read().split('=', 1)[1].strip())
COLUMN_GUIDE = '\n'.join(f'- {en}: {ko}' for ko, en in column_mapping.items())

SQL_SYSTEM = f'''당신은 MySQL 전문가입니다. 질문에 답하는 SELECT 문 하나만 작성하세요.
테이블 `{TABLE_NAME}` 의 정의 :
{schema}

열 이름의 뜻 :
{COLUMN_GUIDE}

규칙 : 기준_년분기_코드(STDR_YYQU_CD)는 연도*10+분기 (예 : 20262 = 2026년 2분기) · 상권 이름은 TRDAR_CD_NM · 결과는 집계해서 50행 이하로'''
REPHRASE_SYSTEM = ('이전 대화를 참고해 사용자의 최신 질문을 대화 기록 없이도 이해되는 구체적인 질문으로 다시 쓰고, '
                   '상권 매출 데이터 조회가 필요한 질문인지 판단하세요.')
REPORT_SYSTEM = '서울시 상권 분석가로서 질문과 SQL 조회 결과만 근거로 마크다운 보고서를 작성하세요. 결과가 비어 있으면 데이터가 없다고 쓰세요.'
GENERAL_SYSTEM = '서울시 상권 분석 도우미입니다. 데이터 조회가 필요 없는 질문에 친절히 답하세요.'
print('SQL 시스템 프롬프트 길이 :', len(SQL_SYSTEM), '자')

SQL 시스템 프롬프트 길이 : 4514 자


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

rephrase_agent = Agent(MODEL_NAME, output_type=RephraseResult, system_prompt=REPHRASE_SYSTEM)
sql_agent = Agent(MODEL_NAME, output_type=SQLResult, system_prompt=SQL_SYSTEM)
report_agent = Agent(MODEL_NAME, output_type=ReportResult, system_prompt=REPORT_SYSTEM)
general_agent = Agent(MODEL_NAME, system_prompt=GENERAL_SYSTEM)                 # output_type 없음 → 글자(str)

In [9]:
# 🧪 OpenAI 대신 PydanticAI 의 FunctionModel(정해 둔 규칙으로 답하는 가짜 모델) — 검사 · 재시도 · 노드 흐름은 진짜와 같다

AREAS = ['성수동카페거리', '흑리단길', '강남역', '홍대입구역']
METRICS = {'여성 매출': 'FML_SELNG_AMT', '남성 매출': 'ML_SELNG_AMT', '20대 매출': 'AGRDE_20_SELNG_AMT', '매출': 'THSMON_SELNG_AMT'}

def last_prompt(messages) -> str:
    return messages[0].parts[-1].content                     # run() 에 넘긴 user_prompt

def reply(info, **fields):
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, fields)])

def fake_rephrase(messages, info):
    prompt = last_prompt(messages)
    history, question = prompt.split('[최신 질문]')
    names = [a for a in AREAS if a in question]
    prev = sorted((a for a in AREAS if a in history and a not in names), key=history.rfind)
    if '거기' in question:                                    # 대화 기록에서 앞서 말한 상권으로 '거기' 를 채움
        names = prev[:1] + names
    elif not names:                                           # 상권을 말하지 않으면 앞 대화의 상권을 이어받음
        names = prev
    year = re.findall(r'(20\d\d)년', question) or re.findall(r'(20\d\d)년', history)[-1:]
    metric = next((m for m in METRICS if m in question), None) or next((m for m in METRICS if m in history), '매출')
    is_data = any(w in question for w in ['매출', '비교', '상권', '분기', '거기'])
    ask = '비교해 주세요' if len(names) > 1 else '알려 주세요'
    q = f"{year[0]}년도 {'와 '.join(names)} 상권의 {metric}액을 {ask}" if is_data and names and year else question.strip()
    return reply(info, rephrased_query=q, is_data_query=is_data)

def fake_sql(messages, info):
    q = last_prompt(messages)
    names = [a for a in AREAS if a in q]; year = int(re.findall(r'(20\d\d)년', q)[0])
    col = next(c for m, c in METRICS.items() if m in q)
    where = ', '.join(f"'{n}'" for n in names)
    sql = (f'SELECT TRDAR_CD_NM, SUM({col}) AS amount FROM {TABLE_NAME} '
           f'WHERE STDR_YYQU_CD BETWEEN {year}1 AND {year}4 AND TRDAR_CD_NM IN ({where}) '
           f'GROUP BY TRDAR_CD_NM ORDER BY amount DESC')
    return reply(info, sql=sql, explanation=f'{year}년 {", ".join(names)} 의 {col} 합계')

def fake_report(messages, info):
    prompt = last_prompt(messages)
    rows = ast.literal_eval(prompt.split('[조회 결과]')[1].strip())   # 가짜 모델이라 결과 글을 다시 읽는다
    if rows and 'error' in rows[0]:
        return reply(info, title='조회 실패', content=rows[0]['error'], conclusion='질문을 바꿔 다시 시도해 주세요.')
    if not rows:
        return reply(info, title='조회 결과 없음', content='해당 조건의 데이터가 테이블에 없습니다.',
                     conclusion='기간(기준 년분기)을 확인해 주세요.')
    lines = '\n'.join(f"| {r['TRDAR_CD_NM']} | {r['amount']:,} |" for r in rows)
    top = rows[0]
    return reply(info, title='상권 매출 비교', content=f'| 상권 | 금액(원) |\n|---|---|\n{lines}',
                 conclusion=f"{top['TRDAR_CD_NM']} 이(가) {top['amount']:,}원으로 가장 높습니다.")

def fake_general(messages, info):
    return ModelResponse(parts=[TextPart('천만에요! 다른 상권도 궁금하면 물어보세요.')])

rephrase_agent = Agent(FunctionModel(fake_rephrase), output_type=RephraseResult, system_prompt=REPHRASE_SYSTEM)
sql_agent = Agent(FunctionModel(fake_sql), output_type=SQLResult, system_prompt=SQL_SYSTEM)
report_agent = Agent(FunctionModel(fake_report), output_type=ReportResult, system_prompt=REPORT_SYSTEM)
general_agent = Agent(FunctionModel(fake_general), system_prompt=GENERAL_SYSTEM)

In [10]:
def history_text(messages) -> str:
    return '\n'.join(f'{type(m).__name__[:-7]}: {m.content}' for m in messages[:-1][-6:])   # 최근 3턴만

async def rephrase_node(state: AnalysisState) -> dict:          # 1. 질문 재구성 (LLM · Agent)
    prompt = f'[대화 기록]\n{history_text(state.messages)}\n[최신 질문]\n{state.messages[-1].content}'
    out = (await rephrase_agent.run(prompt)).output
    print(f'  [1 재구성] {out.rephrased_query} (데이터 조회 : {out.is_data_query})')
    return {'rephrased_query': out.rephrased_query, 'is_data_query': out.is_data_query,
            'sql_query': '', 'sql_result': [], 'final_report': None}          # 📝 앞 턴 값 비우기 (Day 055)

async def sql_node(state: AnalysisState) -> dict:               # 2. SQL 생성 (LLM · Agent)
    out = (await sql_agent.run(state.rephrased_query)).output
    print(f'  [2 SQL] {out.sql}')
    return {'sql_query': out.sql}

async def execute_node(state: AnalysisState) -> dict:           # 3. SQL 실행 (LLM 없음)
    result = await execute_sql_query(state.sql_query)
    if isinstance(result, str):                                  # 📝 실패하면 글자가 온다 → 상태 자료형(list[dict])에 맞게
        print(f'  [3 실행] {result}'); return {'sql_result': [{'error': result}]}
    print(f'  [3 실행] {len(result)}행 {result}')
    return {'sql_result': result}

async def report_node(state: AnalysisState) -> dict:            # 4. 보고서 (LLM · Agent)
    out = (await report_agent.run(f'[질문]\n{state.rephrased_query}\n[조회 결과]\n{state.sql_result}')).output
    text = f'## {out.title}\n\n{out.content}\n\n**결론** : {out.conclusion}'
    return {'final_report': out, 'messages': [AIMessage(text)]}

async def general_node(state: AnalysisState) -> dict:           # 5. 일반 답변 (LLM · 구조화 없음)
    out = (await general_agent.run(state.messages[-1].content)).output
    return {'messages': [AIMessage(out)]}

def route(state: AnalysisState) -> Literal['sql_node', 'general_node']:
    return 'sql_node' if state.is_data_query else 'general_node'

g = StateGraph(AnalysisState)
for name, fn in [('rephrase_node', rephrase_node), ('sql_node', sql_node), ('execute_node', execute_node),
                 ('report_node', report_node), ('general_node', general_node)]:
    g.add_node(name, fn)
g.set_entry_point('rephrase_node')
g.add_conditional_edges('rephrase_node', route, {'sql_node': 'sql_node', 'general_node': 'general_node'})
g.add_edge('sql_node', 'execute_node'); g.add_edge('execute_node', 'report_node')
g.add_edge('report_node', END); g.add_edge('general_node', END)
print(g.compile().get_graph().draw_mermaid().split('graph TD;')[1].split('classDef')[0])


	__start__([<p>__start__</p>]):::first
	rephrase_node(rephrase_node)
	sql_node(sql_node)
	execute_node(execute_node)
	report_node(report_node)
	general_node(general_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> rephrase_node;
	execute_node --> report_node;
	rephrase_node -.-> general_node;
	rephrase_node -.-> sql_node;
	sql_node --> execute_node;
	general_node --> __end__;
	report_node --> __end__;
	


- 📝 필기 상태의 `sql_result: list[dict]` 에 도구 함수의 **오류 글자**(`'SQL 실행 오류: …'`)를 그대로 넣으면 다음 노드가 받을 때 Pydantic 검사에 걸린다(Day 051). 실패는 `[{'error': …}]` 로 감쌌다
- 재구성 노드가 매 턴 `sql_query` · `sql_result` · `final_report` 를 비운다 — 체크포인터가 앞 턴 값을 들고 오기 때문(Day 055)

## 4. 실행 — 한 대화방 4턴 (교재 p.148 예시 질문 포함)

In [11]:
import aiomysql.connection as _aiomysql_conn            # Day 055 : aiomysql + 새 PyMySQL 의 bytes escape 오류 방지 패치
if not callable(_aiomysql_conn.escape_bytes_prefixed):
    _aiomysql_conn.escape_bytes_prefixed = lambda val, mapping=None: f"X'{bytes(val).hex()}'"

from urllib.parse import quote_plus
import uuid
DB_URI = f'mysql://{quote_plus(MYSQL_USER)}:{quote_plus(MYSQL_PASSWORD)}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}'

async def main(questions):
    async with AIOMySQLSaver.from_conn_string(DB_URI) as memory:
        await memory.setup()
        agent = g.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': f'analyst-{uuid.uuid4().hex[:6]}'}}
        for q in questions:
            print(f'사용자 : {q}')
            final = await agent.ainvoke({'messages': [HumanMessage(q)]}, config=config)
            print('AI :\n' + '\n'.join('    ' + line for line in final['messages'][-1].content.splitlines()) + '\n')

In [12]:
# 🧪 OpenAI 대신 PydanticAI 의 FunctionModel(정해 둔 규칙으로 답하는 가짜 모델) — 검사 · 재시도 · 노드 흐름은 진짜와 같다

await main(['2025년 성수동카페거리 여성 매출 알려줘',
            '그럼 거기와 흑리단길 상권을 비교할 경우 2024년도 여성 매출액은 어때?',
            '그럼 2026년으로 다시 비교해줘',
            '고마워요'])

사용자 : 2025년 성수동카페거리 여성 매출 알려줘
  [1 재구성] 2025년도 성수동카페거리 상권의 여성 매출액을 알려 주세요 (데이터 조회 : True)
  [2 SQL] SELECT TRDAR_CD_NM, SUM(FML_SELNG_AMT) AS amount FROM seoul_commercial WHERE STDR_YYQU_CD BETWEEN 20251 AND 20254 AND TRDAR_CD_NM IN ('성수동카페거리') GROUP BY TRDAR_CD_NM ORDER BY amount DESC
  [3 실행] 1행 [{'TRDAR_CD_NM': '성수동카페거리', 'amount': 10662441961}]


Deserializing unregistered type __main__.ReportResult from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'ReportResult')]


AI :
    ## 상권 매출 비교
    
    | 상권 | 금액(원) |
    |---|---|
    | 성수동카페거리 | 10,662,441,961 |
    
    **결론** : 성수동카페거리 이(가) 10,662,441,961원으로 가장 높습니다.

사용자 : 그럼 거기와 흑리단길 상권을 비교할 경우 2024년도 여성 매출액은 어때?
  [1 재구성] 2024년도 성수동카페거리와 흑리단길 상권의 여성 매출액을 비교해 주세요 (데이터 조회 : True)


  [2 SQL] SELECT TRDAR_CD_NM, SUM(FML_SELNG_AMT) AS amount FROM seoul_commercial WHERE STDR_YYQU_CD BETWEEN 20241 AND 20244 AND TRDAR_CD_NM IN ('성수동카페거리', '흑리단길') GROUP BY TRDAR_CD_NM ORDER BY amount DESC


  [3 실행] 0행 []


AI :
    ## 조회 결과 없음
    
    해당 조건의 데이터가 테이블에 없습니다.
    
    **결론** : 기간(기준 년분기)을 확인해 주세요.

사용자 : 그럼 2026년으로 다시 비교해줘
  [1 재구성] 2026년도 성수동카페거리와 흑리단길 상권의 여성 매출액을 비교해 주세요 (데이터 조회 : True)
  [2 SQL] SELECT TRDAR_CD_NM, SUM(FML_SELNG_AMT) AS amount FROM seoul_commercial WHERE STDR_YYQU_CD BETWEEN 20261 AND 20264 AND TRDAR_CD_NM IN ('성수동카페거리', '흑리단길') GROUP BY TRDAR_CD_NM ORDER BY amount DESC
  [3 실행] 2행 [{'TRDAR_CD_NM': '흑리단길', 'amount': 5997291144}, {'TRDAR_CD_NM': '성수동카페거리', 'amount': 4212245255}]
AI :
    ## 상권 매출 비교
    
    | 상권 | 금액(원) |
    |---|---|
    | 흑리단길 | 5,997,291,144 |
    | 성수동카페거리 | 4,212,245,255 |
    
    **결론** : 흑리단길 이(가) 5,997,291,144원으로 가장 높습니다.

사용자 : 고마워요
  [1 재구성] 고마워요 (데이터 조회 : False)


AI :
    천만에요! 다른 상권도 궁금하면 물어보세요.



- 2턴이 교재 p.148 의 예시 그대로다 — '거기' 가 앞 턴의 **성수동카페거리**로 풀려 '2024년도 성수동카페거리와 흑리단길 상권의 여성 매출액을 비교해 주세요' 가 됐다(교재가 기대한 재구성)
- 📝 그런데 테이블에는 **2024년 데이터가 없어**(2025년 1분기 ~ 2026년 2분기) 0행 → 보고서가 '데이터 없음' 으로 끝났다. 교재 예시 질문은 수업 데이터 기간과 맞지 않는다. 보고서 프롬프트에 '결과가 비면 데이터가 없다고 쓰라' 는 규칙이 없으면 LLM 이 숫자를 **지어낼** 수 있다
- 3턴 '2026년으로 다시' 는 상권 · 지표를 기록에서 이어받아 2026년(1~2분기)으로 비교, 4턴 '고마워요' 는 `general_node`
- 📝 첫 턴 실행 중 LangGraph 경고 **`Deserializing unregistered type __main__.ReportResult from checkpoint. This will be blocked in a future version`** — 필기 상태의 `final_report: ReportResult` 처럼 **직접 만든 Pydantic 클래스**를 상태에 넣으면 체크포인터가 저장 · 복원할 때 경고하고, 앞으로는 막힌다. 상태에는 `out.model_dump()` 같은 **dict** 로 넣거나 체크포인터 직렬화 허용 목록에 등록해야 한다
- 🧪 가짜 모델은 정해 둔 질문 꼴에만 맞다. 진짜 LLM 이 어떤 SQL 을 쓰는지(분기 코드 범위 · `GROUP BY` · 상권 이름 오타)는 아래 ⏳ 셀로 확인해야 한다

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

await main(['2025년 성수동카페거리 여성 매출 알려줘',
            '그럼 거기와 흑리단길 상권을 비교할 경우 2025년도 여성 매출액은 어때?',
            '20대 매출이 가장 높은 업종은?',
            '고마워요'])

## 정리 — 교재 p.148 의 4단계와 필기 · 정리본

| 단계 | 교재 | 이 노트 |
|---|---|---|
| ① 재구성 | 이전 대화로 '거기' 를 구체화 | `rephrase_agent` (`RephraseResult` : 질문 + 데이터 조회 여부) → 조회가 아니면 `general_node` |
| ② NL2SQL | 질문 + DB 스키마 → '**SQLite** 쿼리' | `sql_agent` (`SQLResult`) — 📝 필기 DB 는 **MySQL** · 스키마에 한글 뜻 추가 |
| ③ 실행 | '`asyncio.to_thread` 로 안전하게' | 📝 `aiomysql` 은 원래 비동기라 `to_thread` 가 필요 없다 · 대신 **SELECT 검사 + 읽기 전용 계정** |
| ④ 보고서 | 마크다운 보고서 | `report_agent` (`ReportResult` : 제목 · 본문 · 결론) · 빈 결과 처리 |
| 기억 | — | `AIOMySQLSaver` + `thread_id` (Day 055) |